In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")
path2 = "/kaggle/input/q3-stage3-2026/dataset"
print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split

class SUIMDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = os.path.join(root_dir, split)
        self.images_dir = os.path.join(self.root_dir, 'images')
        self.masks_dir = os.path.join(self.root_dir, 'masks')
        self.image_files = sorted(os.listdir(self.images_dir))
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        # Masks usually share the same name or have a specific suffix
        mask_path = os.path.join(self.masks_dir, self.image_files[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") # Load as grayscale for class IDs

        if self.transform:
            image = self.transform(image)

        # Convert mask to tensor and remap labels if necessary
        mask = torch.from_numpy(np.array(mask)).long()
        mask = remap_mask(mask)

        return image, mask

# Define transforms
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# Initialize Dataset and Dataloader
img_path="/kaggle/input/q3-stage3-2026/dataset/images"
mask_path="/kaggle/input/q3-stage3-2026/dataset/masks"


# 2. Split Data
train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    img_path, mask_path, test_size=0.2, random_state=42
)

train_dataset = SUIMDataset(train_imgs, train_masks, transform=transform, mask_transform=transform)
test_dataset  = SUIMDataset(test_imgs,  test_masks,  transform=transform, mask_transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Visualization
images, masks = next(iter(train_loader))
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Image")
plt.imshow(images[0].permute(1, 2, 0).numpy())
plt.subplot(1, 2, 2)
plt.title("Mask")
plt.imshow(masks[0].numpy())
plt.show()












#unknow error here but i coplate the full lab under, I hope it's work for u

In [ ]:
# TO DO
!pip install segmentation-models-pytorch

import segmentation_models_pytorch as smp

# 8 classes as defined in the SUIM dataset table
num_classes = 8

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes,
)


In [ ]:
# TO DO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.float)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch
from torch import nn
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
import torch.optim as optim

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 10
train_losses = []
val_losses = []

for epoch in range(epochs):
    t_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    v_loss = validate(model, val_loader, criterion, device)

    train_losses.append(t_loss)
    val_losses.append(v_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {t_loss:.4f}, Val Loss: {v_loss:.4f}")

# Plotting loss curve
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()